# Clase 229 — Capstone 1: tabular end-to-end

Esqueleto reproducible del capstone: EDA → FE → modelo → tuning → tracking → API stub → dashboard stub → Model Card.

Requisitos mínimos: `numpy`, `pandas`, `scikit-learn`, `optuna`, `joblib`. `mlflow` es opcional (capturado con try/except). Seed = 42 en todo el notebook.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
rng = np.random.default_rng(SEED)
n = 10_000

# Dataset sintético tipo churn: 12 features (num + cat + bool)
tenure       = rng.integers(1, 73, n)
monthly_chg  = rng.normal(70, 25, n).clip(15, 200)
total_chg    = monthly_chg * tenure + rng.normal(0, 50, n)
age          = rng.integers(18, 85, n)
n_services   = rng.integers(1, 8, n)
support_call = rng.poisson(1.5, n)
contract     = rng.choice(['month', 'one_year', 'two_year'], n, p=[0.55, 0.25, 0.20])
payment      = rng.choice(['credit', 'debit', 'bank', 'check'], n)
internet     = rng.choice(['fiber', 'dsl', 'none'], n, p=[0.45, 0.40, 0.15])
gender       = rng.choice(['M', 'F'], n)
paperless    = rng.choice([True, False], n)
auto_pay     = rng.choice([True, False], n, p=[0.4, 0.6])

# Inyectar missing (~3% en monthly_chg)
miss_idx = rng.choice(n, size=int(n*0.03), replace=False)
monthly_chg_obs = monthly_chg.copy(); monthly_chg_obs[miss_idx] = np.nan

# Señal del target: contratos mes-a-mes + soporte alto + sin auto-pay → churn
logit = (
    -1.5
    + 1.4 * (contract == 'month')
    - 0.8 * (contract == 'two_year')
    + 0.30 * support_call
    - 0.025 * tenure
    - 0.6 * auto_pay
    + 0.4 * (internet == 'fiber')
    + rng.normal(0, 0.5, n)
)
p_churn = 1 / (1 + np.exp(-logit))
y = (rng.uniform(0, 1, n) < p_churn).astype(int)

df = pd.DataFrame({
    'tenure': tenure, 'monthly_chg': monthly_chg_obs, 'total_chg': total_chg,
    'age': age, 'n_services': n_services, 'support_call': support_call,
    'contract': contract, 'payment': payment, 'internet': internet,
    'gender': gender, 'paperless': paperless, 'auto_pay': auto_pay,
    'churn': y,
})
print(f'dataset: {df.shape} | churn rate: {df.churn.mean():.3f}')
df.head()

## 1. EDA mínimo

In [ ]:
print('--- dtypes ---'); print(df.dtypes)
print('\n--- missing ---'); print(df.isna().sum()[df.isna().sum() > 0])
print('\n--- balance target ---'); print(df.churn.value_counts(normalize=True).round(3))
print('\n--- describe num ---'); print(df.select_dtypes('number').describe().round(2).T)
print('\n--- correlaciones con churn ---')
print(df.select_dtypes('number').corrwith(df.churn).sort_values(ascending=False).round(3))

## 2. Split estratificado 60/20/20

In [ ]:
X = df.drop(columns=['churn']); y = df['churn']
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=SEED)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=SEED)
for name, part in [('train', X_tr), ('val', X_val), ('test', X_te)]:
    print(f'{name}: {part.shape}')

## 3. Preprocesamiento: ColumnTransformer

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

num_cols = ['tenure', 'monthly_chg', 'total_chg', 'age', 'n_services', 'support_call']
cat_cols = ['contract', 'payment', 'internet', 'gender']
bool_cols = ['paperless', 'auto_pay']

num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])
cat_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                     ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols),
    ('bool', 'passthrough', bool_cols),
])
preprocessor.fit(X_tr)
print('output shape (train):', preprocessor.transform(X_tr).shape)

## 4. Baseline: LogisticRegression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, log_loss

baseline = Pipeline([('pre', preprocessor), ('clf', LogisticRegression(max_iter=1000, random_state=SEED))])
baseline.fit(X_tr, y_tr)
p_val = baseline.predict_proba(X_val)[:, 1]
print(f'baseline ROC-AUC val: {roc_auc_score(y_val, p_val):.4f}')
print(f'baseline log-loss val: {log_loss(y_val, p_val):.4f}')

## 5. Challenger: GradientBoostingClassifier

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gbm = Pipeline([('pre', preprocessor),
                ('clf', GradientBoostingClassifier(random_state=SEED))])
gbm.fit(X_tr, y_tr)
p_val_gbm = gbm.predict_proba(X_val)[:, 1]
print(f'GBM ROC-AUC val: {roc_auc_score(y_val, p_val_gbm):.4f}')

## 6. Tuning con Optuna (20 trials) + MLflow opcional

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    import mlflow
    mlflow.set_experiment('capstone-1-tabular')
    HAS_MLFLOW = True
except Exception:
    HAS_MLFLOW = False
print(f'mlflow disponible: {HAS_MLFLOW}')

def objective(trial):
    params = {
        'max_depth':     trial.suggest_int('max_depth', 2, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators':  trial.suggest_int('n_estimators', 50, 300),
    }
    model = Pipeline([('pre', preprocessor),
                      ('clf', GradientBoostingClassifier(random_state=SEED, **params))])
    model.fit(X_tr, y_tr)
    auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    if HAS_MLFLOW:
        with mlflow.start_run(nested=True):
            mlflow.log_params(params); mlflow.log_metric('val_auc', auc)
    return auc

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=20, show_progress_bar=False)
print(f'best AUC: {study.best_value:.4f}')
print(f'best params: {study.best_params}')

## 7. Modelo final + curvas (ROC, calibración, confusión)

In [ ]:
from sklearn.metrics import roc_curve, confusion_matrix, brier_score_loss
from sklearn.calibration import calibration_curve

final = Pipeline([('pre', preprocessor),
                  ('clf', GradientBoostingClassifier(random_state=SEED, **study.best_params))])
final.fit(X_tr, y_tr)
p_te = final.predict_proba(X_te)[:, 1]

fpr, tpr, _ = roc_curve(y_te, p_te)
frac_pos, mean_pred = calibration_curve(y_te, p_te, n_bins=10)

print(f'test ROC-AUC : {roc_auc_score(y_te, p_te):.4f}')
print(f'test Brier   : {brier_score_loss(y_te, p_te):.4f}')
print(f'test log-loss: {log_loss(y_te, p_te):.4f}')
print('\ncalibration curve (mean_pred → frac_pos):')
for mp, fp in zip(mean_pred.round(2), frac_pos.round(2)):
    print(f'  {mp:.2f} → {fp:.2f}')

## 8. Threshold optimization (F1)

In [ ]:
thresholds = np.linspace(0.05, 0.95, 91)
f1s = [f1_score(y_te, (p_te >= t).astype(int)) for t in thresholds]
best_t = thresholds[int(np.argmax(f1s))]
best_f1 = max(f1s)
print(f'threshold óptimo (F1): {best_t:.2f} → F1 = {best_f1:.4f}')

y_pred = (p_te >= best_t).astype(int)
tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
print(f'matriz confusión @ t={best_t:.2f}: TP={tp} FP={fp} TN={tn} FN={fn}')

## 9. Persistir modelo + schema Pydantic v2 (concepto)

In [ ]:
import joblib, json, tempfile, pathlib
out = pathlib.Path(tempfile.gettempdir()) / 'capstone1_model.joblib'
joblib.dump({'model': final, 'threshold': float(best_t), 'features': list(X.columns)}, out)
print(f'modelo guardado en: {out} ({out.stat().st_size/1024:.1f} KB)')

input_schema = {
    'tenure':       {'type': 'int',   'ge': 1,  'le': 120},
    'monthly_chg':  {'type': 'float', 'ge': 0,  'le': 500},
    'total_chg':    {'type': 'float', 'ge': 0},
    'age':          {'type': 'int',   'ge': 18, 'le': 120},
    'n_services':   {'type': 'int',   'ge': 0,  'le': 20},
    'support_call': {'type': 'int',   'ge': 0,  'le': 100},
    'contract':     {'type': 'Literal[month, one_year, two_year]'},
    'payment':      {'type': 'Literal[credit, debit, bank, check]'},
    'internet':     {'type': 'Literal[fiber, dsl, none]'},
    'gender':       {'type': 'Literal[M, F]'},
    'paperless':    {'type': 'bool'},
    'auto_pay':     {'type': 'bool'},
}
print('\nschema conceptual (mapeará a Pydantic v2 BaseModel):')
print(json.dumps(input_schema, indent=2))

## 10. Stub FastAPI (código — no se ejecuta uvicorn acá)

In [ ]:
fastapi_code = '''
# src/api/main.py
from contextlib import asynccontextmanager
from typing import Literal
from fastapi import FastAPI
from pydantic import BaseModel, Field
import joblib, pandas as pd

ARTIFACT = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    ARTIFACT["bundle"] = joblib.load("models/capstone1_model.joblib")
    yield
    ARTIFACT.clear()

app = FastAPI(title="Capstone 1 - Churn API", version="1.0.0", lifespan=lifespan)

class PredictRequest(BaseModel):
    tenure: int = Field(..., ge=1, le=120)
    monthly_chg: float = Field(..., ge=0, le=500)
    total_chg: float = Field(..., ge=0)
    age: int = Field(..., ge=18, le=120)
    n_services: int = Field(..., ge=0, le=20)
    support_call: int = Field(..., ge=0, le=100)
    contract: Literal["month", "one_year", "two_year"]
    payment: Literal["credit", "debit", "bank", "check"]
    internet: Literal["fiber", "dsl", "none"]
    gender: Literal["M", "F"]
    paperless: bool
    auto_pay: bool

class PredictResponse(BaseModel):
    probability: float
    prediction: int
    threshold: float

@app.get("/health")
def health(): return {"status": "ok"}

@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest) -> PredictResponse:
    b = ARTIFACT["bundle"]
    X = pd.DataFrame([req.model_dump()])[b["features"]]
    p = float(b["model"].predict_proba(X)[0, 1])
    return PredictResponse(probability=p, prediction=int(p >= b["threshold"]), threshold=b["threshold"])
'''
print(fastapi_code)

## 11. Stub Streamlit dashboard con SHAP

In [ ]:
streamlit_code = '''
# app.py
import streamlit as st, pandas as pd, joblib, shap, httpx

st.set_page_config(page_title="Churn dashboard", layout="wide")

@st.cache_resource
def load_model():
    return joblib.load("models/capstone1_model.joblib")

bundle = load_model()
st.title("Capstone 1 - Churn Prediction")

col1, col2 = st.columns(2)
with col1:
    tenure = st.slider("Tenure (meses)", 1, 72, 12)
    monthly = st.number_input("Monthly charge", 15.0, 200.0, 70.0)
    contract = st.selectbox("Contract", ["month", "one_year", "two_year"])
    auto_pay = st.checkbox("Auto pay", value=False)

if st.button("Predict"):
    payload = {"tenure": tenure, "monthly_chg": monthly, "total_chg": monthly*tenure,
               "age": 40, "n_services": 3, "support_call": 1,
               "contract": contract, "payment": "credit", "internet": "fiber",
               "gender": "F", "paperless": True, "auto_pay": auto_pay}
    r = httpx.post("http://api:8000/predict", json=payload, timeout=5).json()
    st.metric("Probabilidad de churn", f"{r['probability']:.1%}")
    st.metric("Predicción", "CHURN" if r['prediction'] else "STAY")

    # SHAP waterfall
    X = pd.DataFrame([payload])[bundle["features"]]
    X_t = bundle["model"].named_steps["pre"].transform(X)
    explainer = shap.TreeExplainer(bundle["model"].named_steps["clf"])
    sv = explainer(X_t)
    st.pyplot(shap.plots.waterfall(sv[0], show=False).figure)
'''
print(streamlit_code)

## 12. Model Card (dict → JSON)

In [ ]:
model_card = {
    'model_name': 'capstone1-churn-gbm',
    'version': '1.0.0',
    'intended_use': 'Predecir probabilidad de churn de clientes telco para campañas de retención.',
    'not_for_use': ['Decisiones de crédito', 'Pricing dinámico', 'Procesos legales o regulatorios'],
    'training_data': {'source': 'sintético tipo Telco Churn',
                       'n_samples': int(len(X_tr)),
                       'features': len(X.columns),
                       'class_balance': round(float(y_tr.mean()), 4)},
    'algorithm': 'GradientBoostingClassifier (sklearn) + Optuna 20 trials',
    'hyperparameters': study.best_params,
    'metrics_test': {
        'roc_auc':  round(float(roc_auc_score(y_te, p_te)), 4),
        'f1':       round(float(best_f1), 4),
        'log_loss': round(float(log_loss(y_te, p_te)), 4),
        'brier':    round(float(brier_score_loss(y_te, p_te)), 4),
        'threshold': round(float(best_t), 2),
    },
    'subgroup_metrics': {
        'gender_M_auc': round(float(roc_auc_score(y_te[X_te.gender == 'M'],
                                                    p_te[X_te.gender.values == 'M'])), 4),
        'gender_F_auc': round(float(roc_auc_score(y_te[X_te.gender == 'F'],
                                                    p_te[X_te.gender.values == 'F'])), 4),
    },
    'limitations': [
        'Entrenado sobre datos sintéticos — no extrapolar a producción sin retraining.',
        'No incluye señal temporal (drift mensual no modelado).',
        'Calibración limitada — usar isotonic/sigmoid si se requiere probabilidad estricta.',
    ],
    'ethical_considerations': 'Revisar paridad de métricas por género/edad antes de retención agresiva.',
    'contact': 'equipo-ds@example.com',
}
print(json.dumps(model_card, indent=2, ensure_ascii=False))

## ✅ Checklist de entregables del capstone

- [ ] Repo público en GitHub con estructura `src/`, `notebooks/`, `tests/`, `MODEL_CARD.md`.
- [ ] `pyproject.toml` + `uv.lock` reproducible.
- [ ] EDA documentado en `notebooks/01_eda.ipynb` + reporte `ydata-profiling`.
- [ ] `ColumnTransformer` con `fit` solo sobre train (sin leakage).
- [ ] MLflow tracking con ≥3 runs (baseline + 2 challengers tuneados).
- [ ] Optuna ≥50 trials sobre el modelo final.
- [ ] Threshold elegido por F1 (o coste-beneficio), no 0.5 default.
- [ ] FastAPI con `/predict` + `/health`, Pydantic v2 con rangos validados, OpenAPI en `/docs`.
- [ ] Latencia P95 `/predict` < 200 ms (medir con `httpx` o `locust`).
- [ ] Streamlit dashboard con form + SHAP waterfall + screenshot en el README.
- [ ] Model Card (Mitchell 2019) con uso intencionado, métricas por subgrupo, limitaciones.
- [ ] `compose.yml` levanta 3 servicios: `mlflow`, `api`, `streamlit`.
- [ ] CI GitHub Actions: `pytest` + smoke test contra `/health`. Badge verde en README.
- [ ] README con instrucciones de reproducción desde clon limpio.